# GLORIA parser walkthrough

This notebook is the practical guide for parsing GLORIA monetary SUT releases in MARIO.

## What this notebook covers

- where the GLORIA release material comes from;
- how the local GLORIA release should be laid out on disk;
- when to pass the release root, the `part_I` folder, or the `GLORIA_MRIOs_*` directory directly;
- how `valuation=`, `regions=`, and `satellites=` change the parse;
- why `dtype` and `cache` matter for large GLORIA runs.

## Relevant source pages

- GLORIA overview: [IELab GLORIA overview](https://ielab.info/resources/gloria/about)
- release notes and supporting material: [IELab GLORIA supporting documents](https://ielab.info/resources/gloria/supportingdocs)

MARIO does not download GLORIA automatically. The expected workflow is to obtain the release locally and then point the parser to the release root, to the `part_I` economic-account folder, or directly to the `GLORIA_MRIOs_*` directory.

## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_gloria(...)`

The current backend supports monetary `SUT` parsing only.

## Key arguments

The key public arguments are:

- `path`: GLORIA release root, the Google Drive `part_I` economic-account folder, or directly the `GLORIA_MRIOs_*` directory;
- `table`: currently only `"SUT"` is supported;
- `valuation`: choose one markup branch such as `basic`, `trade`, `transport`, `taxes`, or `subsidies`;
- `year`: used when the selected root contains more than one GLORIA year;
- `regions`: optional subset of GLORIA region acronyms;
- `satellites`: optional satellite group or row selector;
- `dtype`: numeric storage type, with `float32` as the practical default;
- `cache`: `True` or one explicit path to persist the parsed result.

## Local layout expectation

The parser supports two local layouts.

Compact layout:

- one `GLORIA_MRIOs_*` directory with raw `T`, `Y`, and `V` csv files;
- the matching `GLORIA_ReadMe_*.xlsx` workbook next to that dataset;
- optionally, one matching `GLORIA_SatelliteAccounts_*` directory.

Google Drive layout:

- one `GLORIA_MRIO_Loop*_part_I_MRIOdatabase` folder containing the economic-account dataset;
- one `GLORIA_MRIO_Loop*_part_III_satelliteaccounts` folder containing satellite accounts;
- the matching `GLORIA_ReadMe_*.xlsx` workbook under the same release root.

You can point MARIO to the release root, the `part_I` folder, or directly to one `GLORIA_MRIOs_*` directory.

In [1]:
import mario

## Parse one GLORIA SUT

Use this when you want the default `basic` valuation and the full region set.

In [ ]:
db = mario.parse_gloria(
    path="/path/to/gloria_release",
    year=2023,
)

db

WARNING Parser: GLORIA use block is very large; parsing all regions may require several GB of RAM. Consider the regions=... argument for smaller subsets.
INFO Parser: reading GLORIA transactions from 20260121_120secMother_AllCountries_002_T-Results_2023_060_Markup001(full).csv in CSV chunks.
INFO Parser: reading GLORIA final demand from 20260121_120secMother_AllCountries_002_Y-Results_2023_060_Markup001(full).csv in CSV chunks.
INFO Parser: reading GLORIA value added from 20260121_120secMother_AllCountries_002_V-Results_2023_060_Markup001(full).csv in CSV chunks.
INFO Parser: no GLORIA satellite-account directory found; using empty extensions.
INFO Parser: GLORIA SUT parsed with 164 regions, 120 sectors, 6 value-added rows and 1 satellite rows.
INFO Metadata: initialized.


name = GLORIA SUT 2023 Basic prices
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 120
Commodity = 120
Factor of production = 6
Satellite account = 1
Consumption category = 6
Region = 164

## Restrict the satellite payload

You can keep all satellites, one full row label, or one group name from the GLORIA ReadMe.

In [ ]:
db = mario.parse_gloria(
    path="/path/to/gloria_release",
    satellites="Emissions",
    year = 2023,
)

db

INFO Parser: restricting GLORIA satellites to 5588 rows from request Emissions.
WARNING Parser: GLORIA use block is very large; parsing all regions may require several GB of RAM. Consider the regions=... argument for smaller subsets.
INFO Parser: reading GLORIA transactions from 20260121_120secMother_AllCountries_002_T-Results_2023_060_Markup001(full).csv in CSV chunks.
INFO Parser: reading GLORIA final demand from 20260121_120secMother_AllCountries_002_Y-Results_2023_060_Markup001(full).csv in CSV chunks.
INFO Parser: reading GLORIA value added from 20260121_120secMother_AllCountries_002_V-Results_2023_060_Markup001(full).csv in CSV chunks.
INFO Parser: no GLORIA satellite-account directory found; using empty extensions.
INFO Parser: GLORIA SUT parsed with 164 regions, 120 sectors, 6 value-added rows and 1 satellite rows.
INFO Metadata: initialized.


name = GLORIA SUT 2023 Basic prices
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 120
Commodity = 120
Factor of production = 6
Satellite account = 1
Consumption category = 6
Region = 164

: 

## Practical warnings

- `GLORIA` is currently `SUT`-only in MARIO.
- Full-region parses can be memory-intensive; `regions=`, `dtype="float32"`, and `cache=True` are usually sensible defaults.
- If the local release does not include a complete satellite-account directory, MARIO falls back to empty extensions.